# 04 — Prediction Models

**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`
**Output:** `../outputs/tables/model_summary.csv`, `../outputs/tables/oof_*.csv`

**Description:**
- Run GroupKFold (participant-level) out-of-fold prediction
- Models: baseline (numeric only) vs full (numeric + text PCA)
- Binary outcomes: any_risk, moderate, high_acuity
- Save OOF predictions for downstream evaluation

In [2]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
CRISIS_COL = "crisis_PM_from_full"

N_SPLITS = 5
N_PCS = 20
C_LOGIT = 1.0
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

OUTCOMES = ["any_risk_total_gt0", "moderate_total_ge2", "high_any_item_eq3"]

In [3]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)

assert X_text.shape[0] == len(pm_day), "Row mismatch between data and embeddings"

print("Loaded data:", pm_day.shape)
print("Loaded embeddings:", X_text.shape)

groups = pm_day[PID_COL].astype(str).values

Loaded data: (2511, 66)
Loaded embeddings: (2511, 768)


In [6]:
# =========================
# BUILD NUMERIC BASELINE
# =========================
num_candidates = [c for c in pm_day.columns if c.startswith("dailyPM__") and pd.api.types.is_numeric_dtype(pm_day[c])]
baseline_cols = [CRISIS_COL] + num_candidates

X_num_df = pm_day[baseline_cols].copy()
X_num_df = X_num_df.apply(pd.to_numeric, errors="coerce")
X_num = X_num_df.fillna(X_num_df.mean()).values.astype(float)

print("Baseline features:", X_num.shape[1])

Baseline features: 17


In [8]:
# =========================
# OOF PREDICTION FUNCTION
# =========================
def grouped_oof_logit(X_text, X_num, y, groups, n_splits=5, n_pcs=20, C=1.0, class_weight=None):
    """Grouped (participant) OOF prediction."""
    g = np.asarray(groups).astype(str)
    y = np.asarray(y).astype(int)

    n_groups = len(np.unique(g))
    n_splits_eff = min(n_splits, n_groups)
    gkf = GroupKFold(n_splits=n_splits_eff)

    p_base = np.zeros_like(y, dtype=float)
    p_full = np.zeros_like(y, dtype=float)

    for tr, te in gkf.split(X_num, y, groups=g):
        # Baseline: numeric only
        base = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=C, max_iter=8000, class_weight=class_weight, solver="lbfgs")),
        ])
        base.fit(X_num[tr], y[tr])
        p_base[te] = base.predict_proba(X_num[te])[:, 1]

        # Full: numeric + PCA(text)
        pca = PCA(n_components=min(n_pcs, X_text.shape[1]), random_state=RANDOM_SEED)
        Ttr = pca.fit_transform(X_text[tr])
        Tte = pca.transform(X_text[te])

        Xtr_full = np.hstack([X_num[tr], Ttr])
        Xte_full = np.hstack([X_num[te], Tte])

        full = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=C, max_iter=8000, class_weight=class_weight, solver="lbfgs")),
        ])
        full.fit(Xtr_full, y[tr])
        p_full[te] = full.predict_proba(Xte_full)[:, 1]

    metrics = {
        "pos_rate": float(y.mean()),
        "AUROC_base": float(roc_auc_score(y, p_base)),
        "AUPRC_base": float(average_precision_score(y, p_base)),
        "AUROC_full": float(roc_auc_score(y, p_full)),
        "AUPRC_full": float(average_precision_score(y, p_full)),
    }
    metrics["Delta_AUROC"] = metrics["AUROC_full"] - metrics["AUROC_base"]
    metrics["Delta_AUPRC"] = metrics["AUPRC_full"] - metrics["AUPRC_base"]
    return metrics, p_base, p_full

In [10]:
# =========================
# RUN MODELS
# =========================
all_rows = []

for outcome in OUTCOMES:
    y = pm_day[outcome].astype(int).values
    pos = int(y.sum())

    if pos < 20:
        print(f"Skipping {outcome}: too few positives ({pos})")
        continue

    for model_tag, cw in [("logit", None), ("logit_balanced", "balanced")]:
        label = f"{outcome}__{model_tag}"
        print(f"\n=== {label} ===")

        metrics, p_base, p_full = grouped_oof_logit(
            X_text=X_text, X_num=X_num, y=y, groups=groups,
            n_splits=N_SPLITS, n_pcs=N_PCS, C=C_LOGIT, class_weight=cw
        )

        print(f"  Baseline AUROC/AUPRC: {metrics['AUROC_base']:.3f} / {metrics['AUPRC_base']:.3f}")
        print(f"  Full     AUROC/AUPRC: {metrics['AUROC_full']:.3f} / {metrics['AUPRC_full']:.3f}")
        print(f"  Delta    AUROC/AUPRC: {metrics['Delta_AUROC']:.3f} / {metrics['Delta_AUPRC']:.3f}")

        # Save OOF predictions
        oof = pm_day[[PID_COL, "ema_date"]].copy()
        oof["y"] = y
        oof["p_base"] = p_base
        oof["p_full"] = p_full
        oof_path = os.path.join(OUT_DIR, f"oof_{label}.csv")
        oof.to_csv(oof_path, index=False)

        row = {"outcome": outcome, "model": model_tag, "n": len(y), "positives": pos, **metrics}
        all_rows.append(row)


=== any_risk_total_gt0__logit ===


C:\Users\bb57728\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
C:\Users\bb57728\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
C:\Users\bb57728\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# =========================
# SAVE SUMMARY
# =========================
summary = pd.DataFrame(all_rows)
summary_path = os.path.join(OUT_DIR, "model_summary.csv")
summary.to_csv(summary_path, index=False)

print("\n" + "=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(summary.to_string(index=False))
print("\nSaved:", summary_path)